In [1]:
import os
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm

In [2]:
RAW_DIR = "../data/raw_papers"
OUTPUT_PATH = "../data/processed_chunks/chunks.csv"

os.makedirs("../data/processed_chunks", exist_ok=True)

In [6]:
import re
import xml.etree.ElementTree as ET

def _text_without_formula(node):
    parts = []
    if node.text:
        parts.append(node.text)
    for child in node:
        tag = child.tag.split("}")[-1]
        if tag in {"tex-math", "inline-formula", "disp-formula"}:
            if child.tail:
                parts.append(child.tail)
            continue
        parts.append(_text_without_formula(child))
        if child.tail:
            parts.append(child.tail)
    return "".join(parts)

def parse_pmc_xml(file_path):
    root = ET.parse(file_path).getroot()
    title = root.findtext(".//article-title")
    paragraphs = []

    for p in root.findall(".//body//p"):  # limit to body only
        text = re.sub(r"\s+", " ", _text_without_formula(p)).strip()
        if len(text) > 100:
            paragraphs.append(text)

    return title, paragraphs

In [7]:
records = []

xml_files = os.listdir(RAW_DIR)

for file in tqdm(xml_files):

    file_path = os.path.join(RAW_DIR, file)

    try:
        title, paragraphs = parse_pmc_xml(file_path)

        paper_id = file.replace(".xml", "")

        for i, paragraph in enumerate(paragraphs):

            records.append({
                "paper_id": paper_id,
                "chunk_id": f"{paper_id}_{i}",
                "title": title,
                "text": paragraph
            })

    except Exception as e:
        print("Error parsing", file, e)

  0%|          | 0/300 [00:00<?, ?it/s]

100%|██████████| 300/300 [00:02<00:00, 105.42it/s]


In [8]:
df = pd.DataFrame(records)

df.to_csv(OUTPUT_PATH, index=False)

print("Total chunks:", len(df))

Total chunks: 15241


In [9]:
df.sample(5)

,paper_id,chunk_id,title,text
3037,12960471,12960471_11,Spectrum of Primary Immune Regulatory Disorder...,The frequency of different system involvement ...
8688,12961119,12961119_45,The impact of and changes in the oxidative sta...,Advancements in the diagnostics of oxidative m...
11728,12961856,12961856_94,"A three-arm, parallel group, cluster randomize...",The importance of early intervention in adults...
6892,12960945,12960945_8,Selective sweep probabilities in spatially exp...,The following result (proved in SI Text Sectio...
8476,12961104,12961104_14,Prognostic Value of the Cancer Inflammation Pr...,Overall survival (OS) was defined as the prima...
